# Week 7: Open-source pricer with QLoRA

Same story as week 6, but you **fine-tune a small Hugging Face model** with **QLoRA** instead of the OpenAI fine-tuning API.

**Flow:** Load data → build prompts → **baseline** (base model, no adapter) → **optional QLoRA** (GPU recommended) → evaluate with the shared `pricer` tester.

**Hardware:** Full training needs a CUDA GPU (or use the course Colab notebooks and point `ADAPTER_PATH` at your saved adapter). On CPU/MPS you can still run data prep and a short baseline eval on a tiny slice.

## 1. Setup

Add the week 7 folder to `sys.path` so `pricer` imports work. Put `HF_TOKEN` (and optional `WANDB_API_KEY`) in `.env`.

In [ ]:
import os
import re
import sys

week7_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, week7_root)

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

print("Week7 root:", week7_root)

## 2. Load data and tokenizer

Use `items_lite` for a quicker loop. Swap to `items_full` when you are ready for a serious run.

In [ ]:
LITE_MODE = True
USERNAME = "ed-donner"
dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"

train, val, test = Item.from_hub(dataset_name)
items = train + val + test
for it in items:
    if not it.summary:
        it.summary = it.title

# Ungated instruct model — avoids Llama gating; swap to meta-llama/Llama-3.2-3B if you have access
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Train {len(train):,} | Val {len(val):,} | Test {len(test):,}")

## 3. Token stats and prompts

Match the course: truncate long summaries, round prices on train/val, keep exact prices on test.

In [ ]:
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items, desc="token counts")]
print(f"Avg tokens: {sum(token_counts)/len(token_counts):.1f}, max: {max(token_counts)}")

In [ ]:
CUTOFF = 110
for item in tqdm(train + val, desc="train+val prompts"):
    item.make_prompts(tokenizer, CUTOFF, do_round=True)
for item in tqdm(test, desc="test prompts"):
    item.make_prompts(tokenizer, CUTOFF, do_round=False)

## 4. Baseline: base model (no LoRA)

Greedy generation after `test_prompt()` (ends with `Price is $`). Keep `EVAL_SIZE` small on CPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig


def parse_price(text: str) -> float:
    text = (text or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0


def load_base_model():
    if torch.cuda.is_available():
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=dtype,
            trust_remote_code=True,
        )
        dev = "mps" if torch.backends.mps.is_available() else "cpu"
        model.to(dev)
    model.eval()
    return model


_model = None


def get_model():
    global _model
    if _model is None:
        _model = load_base_model()
    return _model


def hf_generate_price(item: Item) -> str:
    model = get_model()
    prompt = item.test_prompt()
    dev = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    tail = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return str(parse_price(tail))

In [ ]:
EVAL_SIZE = 30  # raise when you have a GPU; full tester default is 200
evaluate(hf_generate_price, test, size=min(EVAL_SIZE, len(test)))

## 5. Optional: QLoRA fine-tuning

Install once: `pip install peft bitsandbytes trl accelerate`.

- Set `RUN_QLORA = True` only on **CUDA** (recommended). The course Colab links in `week7/day3 and 4.ipynb` are the safest path for long jobs.
- `MAX_SAMPLES` keeps a first run small.
- After training, set `ADAPTER_PATH` to your saved adapter and use the next cell to evaluate the merged model.

In [ ]:
RUN_QLORA = False
MAX_SAMPLES = 800
OUTPUT_DIR = os.path.join(week7_root, "community_contributions", "sammyloto", "qlora_adapter")
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

if RUN_QLORA and torch.cuda.is_available():
    from datasets import Dataset
    from peft import LoraConfig
    from transformers import BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer

    def formatting_prompts(examples):
        texts = []
        for p, c in zip(examples["prompt"], examples["completion"]):
            texts.append(p + c)
        return {"text": texts}

    rows = [{"prompt": it.prompt, "completion": it.completion} for it in train[:MAX_SAMPLES]]
    ds = Dataset.from_list(rows).map(formatting_prompts, batched=True)

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    ft_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
    )
    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        max_seq_length=512,
        dataset_text_field="text",
    )
    trainer = SFTTrainer(
        model=ft_model,
        train_dataset=ds,
        peft_config=lora,
        args=args,
        tokenizer=tokenizer,
    )
    trainer.train()
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Saved adapter to", OUTPUT_DIR)
else:
    print("Skipping QLoRA: set RUN_QLORA=True on CUDA, or train in Colab and set ADAPTER_PATH below.")

## 6. Evaluate with a saved adapter (optional)

Point `ADAPTER_PATH` at a folder that contains PEFT weights (local run or downloaded from the Hub).

In [ ]:
ADAPTER_PATH = None  # e.g. OUTPUT_DIR after training, or a Hub id

_ft_model = None


def load_ft_model():
    global _ft_model
    if _ft_model is not None:
        return _ft_model
    if not ADAPTER_PATH:
        return None
    from peft import PeftModel

    if torch.cuda.is_available():
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=dtype, trust_remote_code=True
        )
        base.to("mps" if torch.backends.mps.is_available() else "cpu")
    _ft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    _ft_model.eval()
    return _ft_model


def qlora_pricer(item: Item) -> str:
    m = load_ft_model()
    if m is None:
        return "0"
    prompt = item.test_prompt()
    dev = next(m.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = m.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    tail = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return str(parse_price(tail))


if ADAPTER_PATH:
    evaluate(qlora_pricer, test, size=min(EVAL_SIZE, len(test)))
else:
    print("Set ADAPTER_PATH to evaluate the fine-tuned model.")